In [ ]:
from nba_api.stats.static import teams
from nba_api.stats.endpoints import commonteamroster
from sqlalchemy import create_engine
import pandas as pd
import random
import time
from dotenv import load_dotenv
import os

#helper functions
def normalize_for_postgres(df):
    """Normalize DataFrame column names for PostgreSQL"""
    df = df.copy()
    df.columns = df.columns.str.lower().str.replace(' ', '_')
    return df
    
load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL")
engine = create_engine(DATABASE_URL)

seasons = ['2023-24', '2022-23', '2021-22', '2020-21', '2019-20',
           '2018-19', '2017-18', '2016-17', '2015-16', '2014-15',
           '2013-14', '2012-13', '2011-12', '2010-11', '2009-10']

# Get all NBA teams
nba_teams = teams.get_teams()
print(f"Found {len(nba_teams)} teams in total\n")

# Track unique players
seen_player_ids = set()
all_players = []
team_count = 0

for team in nba_teams:
    team_id = team['id']
    team_name = team['full_name']
    team_count += 1
    
    print(f"[{team_count}/{len(nba_teams)}] Processing team: {team_name} (ID: {team_id})")
    
    try:   
        for season in seasons:
            # Add a sleep to avoid rate limiting
            sleep_time = random.uniform(1.5, 3.0)
            time.sleep(sleep_time)
            
            # Get team roster for that season
            print(f"  Fetching {season} roster...")
            
            try:
                roster = commonteamroster.CommonTeamRoster(
                    team_id=team_id,
                    season=season
                )
                roster_df = roster.get_data_frames()[0]
                
                # Filter out players we've already seen
                new_players_df = roster_df[~roster_df['PLAYER_ID'].isin(seen_player_ids)]
                
                if len(new_players_df) > 0:
                    # Only keep the columns we need for the players table
                    new_players_df = new_players_df[['PLAYER_ID', 'PLAYER']].copy()
                    new_players_df.rename(columns={'PLAYER': 'PLAYER_NAME'}, inplace=True)
                    
                    # Convert to dict records and add to our list
                    new_players = new_players_df.to_dict('records')
                    all_players.extend(new_players)
                    
                    # Update the set of seen player IDs
                    seen_player_ids.update(new_players_df['PLAYER_ID'].tolist())
                    
                    print(f"  ✓ Added {len(new_players)} new players from {season}")
                else:
                    print(f"  - No new players in {season}")
                    
            except Exception as season_error:
                print(f"  ⚠ Error fetching {season} roster: {str(season_error)}")
                continue
        
    except Exception as e:
        print(f"  ⚠ Error processing {team_name}: {str(e)}")
    
    print("-" * 60)

# Create the players DataFrame
print("\n" + "=" * 60)
print("Creating players table...")
print("=" * 60)

players_df = pd.DataFrame(all_players)

# Remove any duplicates (just in case)
players_df = players_df.drop_duplicates(subset=['PLAYER_ID'])

# Sort by player name for easy browsing
players_df = players_df.sort_values('PLAYER_NAME').reset_index(drop=True)

print(f"\nSummary:")
print(f"  Processed {team_count} teams")
print(f"  Found {len(players_df)} unique players across all seasons")
print(f"\nSample of players table:")
print(players_df.head(10))

# Normalize column names before pushing to database
print("\nNormalizing column names for PostgreSQL...")
players_df = normalize_for_postgres(players_df)

# Push to SQL database
print(f"\nPushing to SQL database as 'players' table...")
players_df.to_sql('players', engine, if_exists='replace', index=False)